# DINOv2 — emergent segmentation + a linear probe

**Foundation model** · supports lessons C3 (backbones) and C5/C6 (hands & interaction).

DINOv2 is a self-supervised vision backbone whose features are so structured that objects pop out with **no labels**. We'll (1) run **PCA on its patch tokens** to visualise that, then (2) train a **linear probe** on its CLS feature for classification — the standard way to reuse a frozen foundation backbone.

> **Runtime → T4 GPU** recommended. Meant to run on Colab (downloads DINOv2 weights + data); not pre-executed here.

In [ ]:
!pip -q install transformers
import torch, requests, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
device = "cuda" if torch.cuda.is_available() else "cpu"
proc = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval()
print("DINOv2 ready on", device)

## 1 · Patch-feature PCA — objects emerge without labels

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # two cats
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
inp = proc(images=img, return_tensors="pt").to(device)
with torch.no_grad():
    tokens = dino(**inp).last_hidden_state[0, 1:]               # drop CLS -> (num_patches, dim)
g = int(tokens.shape[0] ** 0.5)
U, S, V = torch.pca_lowrank(tokens, q=3)
pca = (tokens @ V[:, :3]); pca = (pca - pca.min(0).values) / (pca.max(0).values - pca.min(0).values + 1e-6)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(pca.reshape(g, g, 3).cpu()); ax[1].set_title("DINOv2 patch PCA (no labels)"); ax[1].axis("off")
plt.show()

## 2 · Linear probe on CLS features (CIFAR-10)

In [ ]:
from torchvision import datasets
import torch.nn as nn, random
test = datasets.CIFAR10(".", train=False, download=True); random.seed(0)
idxs = random.sample(range(len(test)), 800); feats, labels = [], []
with torch.no_grad():
    for i in idxs:
        im, lab = test[i]
        x = proc(images=im, return_tensors="pt").to(device)
        feats.append(dino(**x).pooler_output.cpu()); labels.append(lab)
X = torch.cat(feats); y = torch.tensor(labels)
clf = nn.Linear(X.shape[1], 10); opt = torch.optim.Adam(clf.parameters(), 1e-3)
for e in range(400):
    opt.zero_grad(); nn.functional.cross_entropy(clf(X[:600]), y[:600]).backward(); opt.step()
print("linear-probe accuracy:", (clf(X[600:]).argmax(-1) == y[600:]).float().mean().item())

### Where to go next
- Run the PCA on **egocentric frames** (EPIC-Kitchens / Ego4D) — hands and manipulated objects separate cleanly.
- Use patch features for open-vocabulary segmentation, or as inputs to a hand/object detector (lessons C5–C6).
- Compare this probe with the **CLIP** probe lab — different pretraining, different features.